### Annotate protein matches with pairwise BLAST
### Julian Moran
### 2026-09-25

In [17]:
import boto3
import io
import glob
import logging
import math
import os
import requests
import s3fs
import subprocess
import tempfile
import time

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from dotenv import load_dotenv
from pathlib import Path

# Env
load_dotenv("../.env", override=True)
REPO_ROOT = os.environ["INSTALL_PATH"]
MINIO_KEY = os.environ["MINIO_KEY"]
MINIO_SECRET = os.environ["MINIO_SECRET"]

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

In [2]:
# ============================================================
#       Args
# ============================================================

# API endpoints


# MinIO
BUCKET = "iei-project"
PREFIX_GOLD = "03_gold/defense_finder/"
PREFIX_SILVER = "02_silver/defense_finder/"
FILE_COMPOSITE = "composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet"

# Dirs
OUT_DIR_PLOT = f"{REPO_ROOT}/vis/plots"

# Check live objects in MinIO silver
client = boto3.client(
    "s3",
    endpoint_url="http://eagle.tcag.ca:9000",
    aws_access_key_id=MINIO_SECRET,
    aws_secret_access_key=MINIO_KEY,
)
response = client.list_objects_v2(
    Bucket=BUCKET,
    Prefix=PREFIX_GOLD
)
live_objects = [
    obj["Key"]
    for obj in response.get("Contents", [])
]
live_objects

['03_gold/defense_finder/composite_score.parquet/_SUCCESS',
 '03_gold/defense_finder/composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet',
 '03_gold/defense_finder/defense_human_domain_annotated.parquet',
 '03_gold/defense_finder/final_output_spark.parquet/_SUCCESS',
 '03_gold/defense_finder/final_output_spark.parquet/part-00000-bb778272-db05-479f-a79a-2d18e89fe363-c000.snappy.parquet',
 '03_gold/defense_finder/griid_gene_subset.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_enriched.manifest.json',
 '03_gold/defense_finder/human_bacteria_structural_analogs_enriched.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/_SUCCESS',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/part-00000-275fe83c-840b-476e-bf63-c47165847863-c000.snappy.parquet']

In [10]:
# ============================================================
#       In
# ============================================================

df_seqs_hs = pl.read_csv(
    f"{REPO_ROOT}/results/iei_human_protein_AAs.tsv",
    separator="\t",
    has_header=True
)

df_seqs_bact = pl.read_csv(
    f"{REPO_ROOT}/results/iei_bact_protein_AAs.tsv",
    separator="\t",
    has_header=True
)

df_comp_score = pl.read_parquet(
    f"s3://{BUCKET}/{PREFIX_GOLD}{FILE_COMPOSITE}",
    storage_options={
        "aws_endpoint_url": "http://eagle.tcag.ca:9000",
        "aws_access_key_id": MINIO_SECRET,
        "aws_secret_access_key": MINIO_KEY,
    }
)

df_comp_score = df_comp_score.rename(
    {
        "defense_uniprot_ac": "bact_uniprot",
        "human_entryId": "hs_uniprot"
    }
)
df_comp_score

bact_uniprot,hs_uniprot,composite_score,foldseek_evalue,pymol_rmsd,tm_score_human,tm_score_bacteria,plddt_human,plddt_bacteria
str,str,f64,f64,f32,f32,f32,f32,f32
"""A0A5C5QGP9""","""A0A024R9P6""",0.2646,0.000087,null,null,null,null,83.480003
"""A0A2R3IRC4""","""A0A0D9SF92""",0.8315,1.2070e-7,0.93,0.8169,0.1028,83.199997,77.449997
"""A0A4D8PF33""","""A0A140VK70""",0.2883,0.003094,15.01,0.2908,0.1539,80.290001,83.050003
"""A0A7S9D461""","""A0A140VK70""",0.8986,5.6920e-19,2.18,0.535,0.7678,80.290001,89.629997
"""A0A2K9LJD6""","""A0A1B0GVC6""",0.2795,0.007862,10.88,0.2428,0.29,67.610001,91.379997
…,…,…,…,…,…,…,…,…
"""A0A1D7XM13""","""Q9H4E3""",0.6409,0.000001,4.93,0.6556,0.327,84.879997,82.360001
"""A0A7D6CQE3""","""Q9H4E3""",0.5814,0.000001,5.77,0.6733,0.3337,84.879997,75.230003
"""A0A5P3ALB7""","""Q9H4E3""",0.8713,1.2930e-15,2.68,0.739,0.4786,84.879997,89.019997


In [12]:
df_seqs_hs

uniprot_accession,sequence
str,str
"""Q16637""","""MAMSSGGSGGGVPEQEDSVLFRRGTGQSDD…"
"""Q8IY37""","""MGKLRRRYNIKGRQQAGPGPSKGPPEPPPV…"
"""Q8NB16""","""MENLKHIITLGQVIHKRCEEMKYCKKQCRR…"
"""Q9NZ01""","""MKHYEVEILDAKTREKLCFLDKVEPHATIA…"
"""Q02156""","""MVVFNGLLKIKICEAVSLKPTAWSLRHAVG…"
…,…
"""J7F7B0""","""SHSMRYFYTAMSRPGRGEPRFITVGYVDDT…"
"""I6MHI2""","""SHSMRYFYTAMSRPGRGEPRFIAVGYVDDT…"
"""D3DP96""","""MKERRASQKLSSKSIMDPNQNVKCKIVVVG…"


In [13]:
# ============================================================
#       Wrangle
# ============================================================

df_seqs = (
    df_comp_score
    .join(
        df_seqs_bact.rename({
            "uniprot_accession": "bact_uniprot",
            "sequence": "bact_seq"
        }),
        on="bact_uniprot",
        how="left"
    )
    .join(
        df_seqs_hs.rename({
            "uniprot_accession": "hs_uniprot",
            "sequence": "hs_seq"
        }),
        on="hs_uniprot",
        how="left"
    )
    .select([
        "bact_uniprot",
        "hs_uniprot",
        "bact_seq",
        "hs_seq"
    ])
)
df_seqs

bact_uniprot,hs_uniprot,bact_seq,hs_seq
str,str,str,str
"""A0A5C5QGP9""","""A0A024R9P6""","""MTTGKLIDRFEGPSGRAVLEEVLLEQKLVL…","""MSRLGALGGARAGLGLLLGTAAGLGFLCLL…"
"""A0A2R3IRC4""","""A0A0D9SF92""","""MGLMDIFRSLIGERDGALSTQQGFQSQPDS…","""PPEENERENGQEILLRLDGSIKGEIRKQAL…"
"""A0A4D8PF33""","""A0A140VK70""","""MKSAFDSVLEMAQAAVLHSMGDTPPTIEDI…","""MPDYLGADQRKTKEDEKDDKPIRALDEGDI…"
"""A0A7S9D461""","""A0A140VK70""","""MDAISDEAQLPHADYAAAWSAIKLDDAVRI…","""MPDYLGADQRKTKEDEKDDKPIRALDEGDI…"
"""A0A2K9LJD6""","""A0A1B0GVC6""","""MNENQYNESAGNGFILSRKGNRKIEGKYVV…","""MDQNNSLPPYAQGLASPQGAMTPGIPIFSP…"
…,…,…,…
"""A0A1D7XM13""","""Q9H4E3""","""MPTNTKEVGLESLIVDYLVNNNGYKQGQNS…","""MAAPEEHDSPTEASQPIVEEEETKTFKDLG…"
"""A0A7D6CQE3""","""Q9H4E3""","""MSKIHDEENFEDFIASHLVEYGGYEYLPSE…","""MAAPEEHDSPTEASQPIVEEEETKTFKDLG…"
"""A0A5P3ALB7""","""Q9H4E3""","""MSSAFDKLARPVQKWIRQKGWRQLRDIQAR…","""MAAPEEHDSPTEASQPIVEEEETKTFKDLG…"


In [31]:
# ============================================================
#       BLAST
# ============================================================

def pairwise_blastp(
    query: str,
    query_ac: str,
    subject: str,
    subject_ac: str
) -> pl.DataFrame:

    with tempfile.TemporaryDirectory() as tmpdir:
        query_path = Path(tmpdir) / "query.fasta"
        subject_path = Path(tmpdir) / "subject.fasta"

        query_path.write_text(f">{query_ac}\n{query}\n")
        subject_path.write_text(f">{subject_ac}\n{subject}\n")
        result = subprocess.run(
            [
                "blastp",
                "-query", str(query_path),
                "-subject", str(subject_path),
                "-outfmt", "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore",
            ],
            capture_output=True,
            text=True,
            check=True,
        )
        columns = [
            "query_ac",
            "subject_ac",
            "pident",
            "length",
            "mismatch",
            "gapopen",
            "qstart",
            "qend",
            "sstart",
            "send",
            "evalue",
            "bitscore",
        ]
        if not result.stdout.strip():
            return pl.DataFrame([
                {
                    column: query_ac if column == "query_ac"
                    else subject_ac if column == "subject_ac"
                    else None
                    for column in columns
                }
            ])
        df_blast = pl.read_csv(
            io.StringIO(result.stdout),
            separator="\t",
            has_header=False,
            new_columns=columns,
        ).sort("bitscore", descending=True)
        return df_blast[0]

def pairwise_blastp_iter(
    df_seqs: pl.DataFrame
) -> pl.DataFrame:
    results = []
    for i in range(len(df_seqs)):
        result = pairwise_blastp(
            query = df_seqs["bact_seq"][i],
            query_ac = df_seqs["bact_uniprot"][i],
            subject = df_seqs["hs_seq"][i],
            subject_ac = df_seqs["hs_uniprot"][i]
        )  
        results.append(result)

    df_blast = pl.concat(results)
    df_ann = df_seqs.join(
        df_blast,
        left_on=["bact_uniprot", "hs_uniprot"],
        right_on=["query_ac", "subject_ac"],
        how="left"
    )
    return df_ann

tmp = df_seqs[:100]
tmp = pairwise_blastp_iter(tmp)
tmp

bact_uniprot,hs_uniprot,bact_seq,hs_seq,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore
str,str,str,str,f64,i64,i64,i64,i64,i64,i64,i64,f64,f64
"""A0A5C5QGP9""","""A0A024R9P6""","""MTTGKLIDRFEGPSGRAVLEEVLLEQKLVL…","""MSRLGALGGARAGLGLLLGTAAGLGFLCLL…",31.25,48,26,2,21,67,261,302,0.07,20.4
"""A0A2R3IRC4""","""A0A0D9SF92""","""MGLMDIFRSLIGERDGALSTQQGFQSQPDS…","""PPEENERENGQEILLRLDGSIKGEIRKQAL…",35.294,68,43,1,900,966,26,93,4.7800e-11,46.2
"""A0A4D8PF33""","""A0A140VK70""","""MKSAFDSVLEMAQAAVLHSMGDTPPTIEDI…","""MPDYLGADQRKTKEDEKDDKPIRALDEGDI…",41.176,17,10,0,621,637,333,349,1.4,18.1
"""A0A7S9D461""","""A0A140VK70""","""MDAISDEAQLPHADYAAAWSAIKLDDAVRI…","""MPDYLGADQRKTKEDEKDDKPIRALDEGDI…",25.444,169,87,7,57,211,212,355,9.1200e-8,38.9
"""A0A2K9LJD6""","""A0A1B0GVC6""","""MNENQYNESAGNGFILSRKGNRKIEGKYVV…","""MDQNNSLPPYAQGLASPQGAMTPGIPIFSP…",40.0,10,6,0,188,197,194,203,3.4,13.5
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""G2LKY9""","""D6RAF8""","""MTDSPIVEMRFYLSGERLPVDHGYLLYAAL…","""MSEEQFGGDGAAAAATAAVGGSAGEQEGAM…",30.769,13,9,0,169,181,194,206,9.6,11.9
"""A0A7H1RXG3""","""H3BLU2""","""MYISKLVIEGYRCFNEKTEIPLNEGLTVIL…","""MRLLCLLPTGLPVRSVDFNRGTDNITVRQG…",null,null,null,null,null,null,null,null,null,null
"""A0A1X9MBY8""","""H7C5K0""","""MLDVGEIVKGPFWSEIVEIKKCELIDDALY…","""DYGTKGDSPLHSIRWLRVILDEGHAIRNPN…",36.709,79,45,2,507,580,347,425,3.0200e-11,52.4


In [25]:
df_seqs

bact_uniprot,hs_uniprot,bact_seq,hs_seq
str,str,str,str
"""A0A5C5QGP9""","""A0A024R9P6""","""MTTGKLIDRFEGPSGRAVLEEVLLEQKLVL…","""MSRLGALGGARAGLGLLLGTAAGLGFLCLL…"
"""A0A2R3IRC4""","""A0A0D9SF92""","""MGLMDIFRSLIGERDGALSTQQGFQSQPDS…","""PPEENERENGQEILLRLDGSIKGEIRKQAL…"
"""A0A4D8PF33""","""A0A140VK70""","""MKSAFDSVLEMAQAAVLHSMGDTPPTIEDI…","""MPDYLGADQRKTKEDEKDDKPIRALDEGDI…"
"""A0A7S9D461""","""A0A140VK70""","""MDAISDEAQLPHADYAAAWSAIKLDDAVRI…","""MPDYLGADQRKTKEDEKDDKPIRALDEGDI…"
"""A0A2K9LJD6""","""A0A1B0GVC6""","""MNENQYNESAGNGFILSRKGNRKIEGKYVV…","""MDQNNSLPPYAQGLASPQGAMTPGIPIFSP…"
…,…,…,…
"""A0A1D7XM13""","""Q9H4E3""","""MPTNTKEVGLESLIVDYLVNNNGYKQGQNS…","""MAAPEEHDSPTEASQPIVEEEETKTFKDLG…"
"""A0A7D6CQE3""","""Q9H4E3""","""MSKIHDEENFEDFIASHLVEYGGYEYLPSE…","""MAAPEEHDSPTEASQPIVEEEETKTFKDLG…"
"""A0A5P3ALB7""","""Q9H4E3""","""MSSAFDKLARPVQKWIRQKGWRQLRDIQAR…","""MAAPEEHDSPTEASQPIVEEEETKTFKDLG…"
